# Day 05 – Memory, Vector Databases & RAG

## Drive Ready – AI Agents | OPC Initiative 2026

### Objective
Build an intelligent AI agent using conversation memory, embeddings, ChromaDB, chunking and Retrieval-Augmented Generation (RAG).

In [1]:
!pip install -q langchain langchain-groq langgraph chromadb langchain-text-splitters pydantic

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
tweepy 4.17.0 requires oauthlib<4,>=3.2.0, but you have oauthlib 4.0.1 which is incompatible.


In [2]:
import chromadb
import langchain
import langgraph

from langchain_groq import ChatGroq
from langchain_text_splitters import RecursiveCharacterTextSplitter

print("All required libraries imported successfully!")

All required libraries imported successfully!


In [3]:
import os
from google.colab import userdata
from langchain_groq import ChatGroq

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.2,
    max_retries=3
)

print(llm.invoke("Explain RAG in one sentence.").content)

RAG (Retrieval‑Augmented Generation) is a technique that blends a document‑retrieval step with a generative language model, so the model can pull in relevant external information before producing a context‑aware answer.


In [4]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

memory = InMemorySaver()

agent = create_agent(
    model=llm,
    tools=[],
    system_prompt="You are a friendly Sweet Crumbs Bakery assistant.",
    checkpointer=memory
)

print("Memory-enabled agent created successfully!")

Memory-enabled agent created successfully!


In [5]:
def chat(message, thread_id):

    config = {
        "configurable": {
            "thread_id": thread_id
        }
    }

    response = agent.invoke(
        {
            "messages": [
                {"role": "user", "content": message}
            ]
        },
        config=config
    )

    print("Assistant:", response["messages"][-1].content)

In [6]:
chat("My name is Priya.", "customer_1")

chat("I like chocolate cakes.", "customer_1")

chat("What is my name and favourite cake?", "customer_1")

Assistant: Hello Priya! 👋 Welcome to Sweet Crumbs Bakery. How can I sprinkle a little sweetness into your day today? Whether you’re craving a fresh loaf, a custom cake, or just need some baking tips, I’m here to help!
Assistant: That’s a delicious choice! 🎂✨

At Sweet Crumbs Bakery, we have a few chocolate‑cake specialties you might love:

| Cake | Description | Size | Price |
|------|-------------|------|-------|
| **Classic Dark Chocolate Velvet** | Rich, moist, with a silky chocolate ganache frosting | 8” | $28 |
| **Triple‑Layer Chocolate Fudge** | Three layers of fudgy cake, chocolate‑cream filling, topped with chocolate shavings | 10” | $38 |
| **Chocolate‑Berry Surprise** | Dark chocolate cake with a raspberry‑chocolate swirl, finished with a chocolate glaze | 8” | $30 |
| **Vegan Dark Chocolate Delight** | Dairy‑free, gluten‑free, made with almond milk and coconut oil | 8” | $32 |

**Got a special occasion in mind?**  
- **Birthday** – we can add a personalized topper.  
- **We

In [7]:
chat("My name is Anjali.", "customer_2")

chat("I like cupcakes.", "customer_2")

chat("What is my name and favourite cake?", "customer_2")

Assistant: Hello Anjali! 👋 Welcome to Sweet Crumbs Bakery. How can I help you today? Whether you’re looking for a special treat, have a question about our menu, or need a recipe tip, I’m here to make your day a little sweeter!
Assistant: That’s fantastic, Anjali! 🎉 Cupcakes are our specialty. Do you have a favorite flavor or a particular occasion in mind? I can suggest a few of our most popular varieties or even help you pick the perfect frosting and decorations. Let me know what you’re craving!
Assistant: Your name is **Anjali**, and your favorite cake is **cupcakes**! 🎂✨ If you’d like to explore some of our best cupcake flavors or need a recipe, just let me know!


In [8]:
chat("Do you remember my name?", "customer_1")

chat("Do you remember my name?", "customer_2")

Assistant: Yes, I remember! Your name is **Priya**. If there’s anything else you’d like to chat about or order, just let me know!
Assistant: Yes, I remember! Your name is **Anjali**. If there’s anything else you’d like to chat about—cupcake flavors, orders, or baking tips—just let me know!


In [9]:
from chromadb.utils.embedding_functions import DefaultEmbeddingFunction

embedding_function = DefaultEmbeddingFunction()

sentences = [
    "Chocolate cake",
    "Vanilla cake",
    "Home delivery"
]

embeddings = embedding_function(sentences)

print("Number of embeddings:", len(embeddings))
print("Vector dimensions:", len(embeddings[0]))
print("Sample vector:", embeddings[0][:10])

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:00<00:00, 104MiB/s]


Number of embeddings: 3
Vector dimensions: 384
Sample vector: [-0.01584118  0.03076809 -0.00851863  0.06349499 -0.06835908  0.07806112
  0.08664065 -0.05573478  0.02715896 -0.01711126]


In [10]:
import numpy as np

def cosine_similarity(a, b):
    return np.dot(a, b) / (
        np.linalg.norm(a) * np.linalg.norm(b)
    )

print("Cake vs Cake:",
      cosine_similarity(embeddings[0], embeddings[1]))

print("Cake vs Delivery:",
      cosine_similarity(embeddings[0], embeddings[2]))

Cake vs Cake: 0.7654911
Cake vs Delivery: 0.2713619


In [11]:
import chromadb

client = chromadb.Client()

faq = client.get_or_create_collection(
    name="sweet_crumbs_faq"
)

print("ChromaDB collection created successfully!")

ChromaDB collection created successfully!


In [12]:
import chromadb

client = chromadb.Client()

faq = client.get_or_create_collection(
    name="sweet_crumbs_faq"
)

print("ChromaDB collection created successfully!")

ChromaDB collection created successfully!


In [13]:
faq.upsert(
    ids=["delivery", "payment", "photo_cake"],

    documents=[
        "We deliver within 5 km of the bakery. Delivery is Rs 50, free for orders above Rs 1,000.",

        "We accept UPI and cash. Orders above Rs 1,000 need a 50% advance by UPI.",

        "Photo cakes and custom designs need 48 hours' notice."
    ]
)

print("Documents stored:", faq.count())

Documents stored: 3


In [14]:
results = faq.query(
    query_texts=[
        "Can you send the cake to my house?"
    ],
    n_results=2
)

print(results["documents"][0])

["Photo cakes and custom designs need 48 hours' notice.", 'We deliver within 5 km of the bakery. Delivery is Rs 50, free for orders above Rs 1,000.']


In [15]:
questions = [
    "Can I pay using PhonePe?",
    "How early should I order a photo cake?",
    "Is home delivery free?"
]

for question in questions:

    result = faq.query(
        query_texts=[question],
        n_results=1
    )

    print("Question:", question)
    print("Retrieved:", result["documents"][0][0])
    print("-" * 50)

Question: Can I pay using PhonePe?
Retrieved: We accept UPI and cash. Orders above Rs 1,000 need a 50% advance by UPI.
--------------------------------------------------
Question: How early should I order a photo cake?
Retrieved: Photo cakes and custom designs need 48 hours' notice.
--------------------------------------------------
Question: Is home delivery free?
Retrieved: We deliver within 5 km of the bakery. Delivery is Rs 50, free for orders above Rs 1,000.
--------------------------------------------------


In [16]:
bakery_document = """
Sweet Crumbs Bakery Policies

Delivery Policy:
We deliver within 5 km of the bakery.
Delivery costs Rs 50 and is free for orders above Rs 1000.

Payment Policy:
We accept UPI and cash.
Orders above Rs 1000 require a 50% advance payment.

Customization Policy:
Photo cakes and custom designs require 48 hours notice.
Eggless cakes are available on request.

Cancellation Policy:
Orders can be cancelled up to 24 hours before delivery.
Customized cakes cannot be cancelled after preparation begins.

Storage Instructions:
Cream cakes must be refrigerated.
Consume cakes within 24 hours for the best quality.
"""

In [17]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=30
)

chunks = splitter.split_text(bakery_document)

print("Total chunks:", len(chunks))

for i, chunk in enumerate(chunks):
    print(f"\nChunk {i+1}:")
    print(chunk)

Total chunks: 5

Chunk 1:
Sweet Crumbs Bakery Policies

Delivery Policy:
We deliver within 5 km of the bakery.
Delivery costs Rs 50 and is free for orders above Rs 1000.

Chunk 2:
Payment Policy:
We accept UPI and cash.
Orders above Rs 1000 require a 50% advance payment.

Chunk 3:
Customization Policy:
Photo cakes and custom designs require 48 hours notice.
Eggless cakes are available on request.

Chunk 4:
Cancellation Policy:
Orders can be cancelled up to 24 hours before delivery.
Customized cakes cannot be cancelled after preparation begins.

Chunk 5:
Storage Instructions:
Cream cakes must be refrigerated.
Consume cakes within 24 hours for the best quality.


In [18]:
policy_collection = client.get_or_create_collection(
    name="sweet_crumbs_policies"
)

policy_collection.upsert(
    ids=[f"chunk_{i}" for i in range(len(chunks))],
    documents=chunks
)

print("Stored chunks:", policy_collection.count())

Stored chunks: 5


In [19]:
results = policy_collection.query(
    query_texts=["Can I cancel my cake order?"],
    n_results=2
)

for document in results["documents"][0]:
    print(document)
    print("-" * 40)

Cancellation Policy:
Orders can be cancelled up to 24 hours before delivery.
Customized cakes cannot be cancelled after preparation begins.
----------------------------------------
Customization Policy:
Photo cakes and custom designs require 48 hours notice.
Eggless cakes are available on request.
----------------------------------------


In [20]:
def rag_answer(question):

    results = policy_collection.query(
        query_texts=[question],
        n_results=2
    )

    context = "\n\n".join(
        results["documents"][0]
    )

    prompt = f"""
    You are a Sweet Crumbs Bakery assistant.

    Answer the customer question using only
    the provided context.

    If the information is not available,
    say you do not have that information.

    Context:
    {context}

    Customer question:
    {question}
    """

    response = llm.invoke(prompt)

    print("Question:", question)
    print("Answer:", response.content)

In [21]:
rag_answer("Can you deliver cakes to my home?")

rag_answer("Can I cancel my customized cake?")

rag_answer("Do you accept UPI payments?")

rag_answer("How should I store a cream cake?")

Question: Can you deliver cakes to my home?
Answer: I’m sorry, but I don’t have that information.
Question: Can I cancel my customized cake?
Answer: You can cancel a customized cake only before the preparation begins. Once the cake has started being made, it cannot be cancelled.
Question: Do you accept UPI payments?
Answer: Yes, we accept UPI payments.
Question: How should I store a cream cake?
Answer: Cream cakes should be refrigerated. For best quality, consume them within 24 hours.


In [22]:
rag_answer("Does your bakery have branches in Hyderabad?")

Question: Does your bakery have branches in Hyderabad?
Answer: I’m sorry, but I don’t have that information.


In [23]:
from langchain.tools import tool

@tool
def search_bakery_knowledge(question: str) -> str:
    """Search bakery policies about delivery, payment,
    customization, cancellation and storage."""

    results = policy_collection.query(
        query_texts=[question],
        n_results=2
    )

    return "\n\n".join(
        results["documents"][0]
    )

In [24]:
SYSTEM = """
You are a friendly Sweet Crumbs Bakery assistant.

RULES:
1. Remember customer details within the conversation.
2. Use search_bakery_knowledge for bakery policy questions.
3. Answer using retrieved information.
4. Never invent business policies.
5. If information is unavailable, say so.
6. Be friendly and concise.
"""

final_agent = create_agent(
    model=llm,
    tools=[search_bakery_knowledge],
    system_prompt=SYSTEM,
    checkpointer=InMemorySaver()
)

print("Sweet Crumbs RAG Agent is ready!")

Sweet Crumbs RAG Agent is ready!


In [25]:
def bakery_chat(message, thread_id="customer_1"):

    response = final_agent.invoke(
        {
            "messages": [
                {"role": "user", "content": message}
            ]
        },
        config={
            "configurable": {
                "thread_id": thread_id
            }
        }
    )

    print("Sweet Crumbs:", response["messages"][-1].content)

In [26]:
bakery_chat("My name is Priya.")

bakery_chat("Can you deliver a cake to my house?")

bakery_chat("Do you accept UPI?")

bakery_chat("What is my name?")

Sweet Crumbs: Hello Priya! 👋 How can Sweet Crumbs help you today?
Sweet Crumbs: Absolutely! We offer delivery for orders within 5 km of the bakery. The delivery fee is Rs 50, but it’s free if your order totals over Rs 1,000. Let me know what you’d like, and I’ll arrange it for you.
Sweet Crumbs: Yes, we accept UPI! For orders over Rs 1,000 we’ll need a 50 % advance payment, and the rest can be paid on delivery (or via UPI as well). Let me know the details of your order, and I’ll set everything up.
Sweet Crumbs: Your name is Priya.


In [27]:
print("WELCOME TO SWEET CRUMBS AI")
print("Type exit to stop.\n")

while True:

    message = input("You: ")

    if message.lower().strip() == "exit":
        print("Thank you for visiting Sweet Crumbs!")
        break

    if message.strip():
        bakery_chat(message, "live_customer")

WELCOME TO SWEET CRUMBS AI
Type exit to stop.

You: exit
Thank you for visiting Sweet Crumbs!
